In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
#==========================================================
# Load Packages 
#==========================================================
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import sys
sys.path.append('/project/IZZY/MolRepres/Methods/')                 # Path where the model is located 
#from dig.threedgraph.dataset import QM93D                          # Dataset utilities for 3D molecular graphs (QM9 with 3D coordinates)
from torch_geometric.datasets import QM9                            # Loading the QM9 dataset from Torch-Geometric
from torch_geometric.loader import DataLoader                       # DataLoader to batch and shuffle molecular graph data
from torch.optim import Adam                                        # Optimizer (Adam) and learning rate scheduler (StepLR)
from torch.optim.lr_scheduler import StepLR                         # Progress bar for training/evaluation loops
from tqdm import tqdm                                               # Unit conversion constants (Hartree, eV, Bohr, Angstrom) from ASE
from ase.units import Hartree, eV, Bohr, Ang                        # TensorBoard writer for tracking metrics and visualizing training progress
from torch.utils.tensorboard import SummaryWriter                   # TensorBoard writer for tracking metrics and visualizing training progress 
from model import SphereNet_model                                   # 3D GNN model implementation (SphereNet) from DIG (Deep Graph Library for 3D Graphs)
from train_eval import train_epoch, evaluate                        # Importing the training and evaluating function

In [3]:
#==========================================================
# GPU Device
#==========================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # This line checks if a CUDA-enabled GPU is available. If yes, computations will be performed on the GPU for faster training. Otherwise, it falls back to using the CPU.
# Print which device is being used 
print("Using device:", device)

Using device: cuda


In [4]:
#==============================================================
# Load dataset and split
#==============================================================

# Load the QM9 dataset 
dataset = QM9(root='data/QM9')

# Count how many molecular graphs are in the dataset
num_mols = len(dataset)
print(f"Total QM9 molecules: {num_mols}")

# Define the desired split sizes for training, validation, and testing
train_size, valid_size = 110000, 10000
# Ensure the sum of train and validation sizes does not exceed dataset size
assert train_size + valid_size < num_mols, "Split sizes too large for dataset"

# Randomly shuffle all molecule indices
perm = torch.randperm(num_mols, generator=torch.Generator().manual_seed(42))

# Use the shuffled indices to create non-overlapping subsets
train_idx = perm[:train_size]
valid_idx = perm[train_size:train_size+valid_size]
test_idx  = perm[train_size+valid_size:]

# Build subsets based on the index splits
train_dataset = dataset[train_idx]
valid_dataset = dataset[valid_idx]
test_dataset  = dataset[test_idx]

Total QM9 molecules: 130831


In [5]:
#======================================================================
# Define the QM9 Targets
#======================================================================
PROPERTY_NAMES = [
    'μ (D)', 'α (Ang³)', 'ε_HOMO (eV)', 'ε_LUMO (eV)', 'Δε (eV)', '⟨R²⟩ (Ang²)',
    'ZPVE (eV)', 'U₀ (eV)', 'U (eV)', 'H (eV)', 'G (eV)', 'c_v', 'U₀_atom',
    'U_atom', 'H_atom', 'G_atom', 'A (GHz)', 'B (GHz)', 'C (GHz)'
] 

# ======================================================================
# QM9 Unit Conversion Factors
# ======================================================================
def get_qm9_conversions_tensor(device):
    return torch.tensor([
        1.0, Bohr**3 / Ang**3, Hartree / eV, Hartree / eV, Hartree / eV,
        Bohr**2 / Ang**2, Hartree / eV, Hartree / eV, Hartree / eV,
        Hartree / eV, Hartree / eV, 1.0, 1.0, Hartree / eV,
        Hartree / eV, Hartree / eV, 1.0, 1.0, 1.0
    ], dtype=torch.float, device=device)

In [6]:
#========================================================================
# Normalize all QM9 targets
#========================================================================
y_raw_all = dataset.data.y.clone().cpu()
conversions_cpu = get_qm9_conversions_tensor('cpu')
y_conv_all = y_raw_all * conversions_cpu.unsqueeze(0)

norm_stats = {'mean': [], 'std': []}
y_norm_all = torch.zeros_like(y_conv_all)

for i in range(y_conv_all.shape[1]):
    train_y_cpu = y_conv_all[train_idx.cpu(), i]
    mean_i = float(train_y_cpu.mean().item())
    std_i = float(train_y_cpu.std().item()) if train_y_cpu.std().item() != 0 else 1.0
    y_norm_all[:, i] = (y_conv_all[:, i] - mean_i) / std_i
    norm_stats['mean'].append(mean_i)
    norm_stats['std'].append(std_i)

dataset.data.y = y_norm_all.to(torch.float)
print("Normalization complete for all targets.")

Normalization complete for all targets.


/home/ismail/dig_envi/lib/python3.9/site-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


In [ ]:
#==============================================================================================
# Trainig Loop
#==============================================================================================

def main():
    #------------------------------
    # Hyperparameters
    #------------------------------
    epochs = 1000                         # number of training epochs
    batch_size = 128                      # batch size for training
    vt_batch_size = 256                   # batch size for validation
    lr = 1e-5                             # learning rate
    lr_decay_step = 50                    # steps after which LR is decayed
    lr_decay_factor = 0.5                 # factor to decay learning rate
    weight_decay = 1e-4                   # L2 regularization
    save_dir = 'checkpoints_SphereNet'    # directory to save model checkpoints 
    log_dir = 'logs_SphereNet'            # TensorBoard logs directory

    #Create directories if they don't exist
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)

    # TensorBoard write for visualization of metrics
    writer = SummaryWriter(log_dir=log_dir)

    #-----------------------------
    # Dataloaders 
    #-----------------------------

    # Shuffled batches for training; sequential batches for validation/testing
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(valid_dataset, batch_size=vt_batch_size, shuffle=False)
    test_loader  = DataLoader(test_dataset, batch_size=vt_batch_size, shuffle=False)

    #----------------------------------------
    # Model
    #----------------------------------------
    # SphereNet model
    model = SphereNet_model().to(device)   

    #----------------------------------------
    # Optimizer and Scheduler
    #----------------------------------------
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = StepLR(optimizer, step_size=lr_decay_step, gamma=lr_decay_factor)

    # Track best validation/test performance
    best_mean_val=float('inf')
    best_val = [float('inf')] * len(PROPERTY_NAMES)
    best_test = [float('inf')] * len(PROPERTY_NAMES)

    # Print total number of trainabel parameters and target property
    print("#params:", sum(p.numel() for p in model.parameters()))
    print("Training for all 19 targets") 
    
    #--------------------------------------------
    # Training Loop
    #--------------------------------------------
    for epoch in range(1, epochs + 1):
        print(f"\n=== Epoch {epoch} ===")

        #Train for one epoch
        train_loss = train_epoch(model, train_loader, optimizer, device)

        # Evaluate on validation and test sets (MAE in original units)
        val_mae = evaluate(model, val_loader, device, norm_stats['mean'], norm_stats['std'])
        test_mae = evaluate(model, test_loader, device, norm_stats['mean'], norm_stats['std'])

        # Print epoch metrics
        print(f"Train loss (MSE): {train_loss:.6f}")
        for i, prop in enumerate(PROPERTY_NAMES):
            print(f"  {prop:15s} | Val MAE: {val_mae[i]:.6f} | Test MAE: {test_mae[i]:.6f}")
            writer.add_scalar(f'val_mae/{prop}', val_mae[i], epoch)
            writer.add_scalar(f'test_mae/{prop}', test_mae[i], epoch)
        #---------------------------------------------
        # Save checkpoint if validation improves 
        #---------------------------------------------
        mean_val_mae = sum(val_mae) / len(val_mae)
        
        # If mean validation MAE improved → save one best model
        if mean_val_mae < best_mean_val:
            best_mean_val = mean_val_mae
            best_val = val_mae
            best_test = test_mae
        
            save_path = os.path.join(save_dir, 'best_model.pt')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val': best_val,
                'best_test': best_test,
                'best_mean_val': best_mean_val
            }, save_path)
        
            print(f" Saved best overall model (mean validation MAE improved to {best_mean_val:.4f})")

        # Update learning rate according to scheduler
        scheduler.step()
    # Close TensorBoard writer
    writer.close()
    print("\nFinished training.")
    print("Best validation and test MAEs per property:")
    for prop, v, t in zip(PROPERTY_NAMES, best_val, best_test):
        print(f"  {prop:15s} | Best validation MAE: {v:.6f} | Test MAE at best val: {t:.6f}")

    #print("Best validation MAE:", best_val)
    #print("Test MAE at best val:", best_test)

# Run the training loop
if __name__ == "__main__":
    main()

#params: 1913158
Training for all 19 targets

=== Epoch 1 ===


Train loss (MSE): 1.273300
  μ (D)           | Val MAE: 1.007827 | Test MAE: 1.018092
  α (Ang³)        | Val MAE: 0.570787 | Test MAE: 0.558645
  ε_HOMO (eV)     | Val MAE: 10.238638 | Test MAE: 10.167791
  ε_LUMO (eV)     | Val MAE: 18.856491 | Test MAE: 18.974936
  Δε (eV)         | Val MAE: 19.493639 | Test MAE: 19.302847
  ⟨R²⟩ (Ang²)     | Val MAE: 40.005650 | Test MAE: 39.776661
  ZPVE (eV)       | Val MAE: 7.755254 | Test MAE: 7.596560
  U₀ (eV)         | Val MAE: 13518.463867 | Test MAE: 13246.689453
  U (eV)          | Val MAE: 13303.551758 | Test MAE: 13087.410156
  H (eV)          | Val MAE: 14279.024414 | Test MAE: 13956.439453
  G (eV)          | Val MAE: 13138.654297 | Test MAE: 12956.605469
  c_v             | Val MAE: 1.525141 | Test MAE: 1.495006
  U₀_atom         | Val MAE: 3.600397 | Test MAE: 3.496460
  U_atom          | Val MAE: 104.869385 | Test MAE: 102.856941
  H_atom          | Val MAE: 98.817001 | Test MAE: 96.069046
  G_atom          | Val MAE: 92.129166 | T

Train loss (MSE): 0.470743
  μ (D)           | Val MAE: 0.925944 | Test MAE: 0.943431
  α (Ang³)        | Val MAE: 0.512519 | Test MAE: 0.500446
  ε_HOMO (eV)     | Val MAE: 8.849081 | Test MAE: 8.785630
  ε_LUMO (eV)     | Val MAE: 15.688053 | Test MAE: 15.851412
  Δε (eV)         | Val MAE: 16.910831 | Test MAE: 16.837452
  ⟨R²⟩ (Ang²)     | Val MAE: 35.785778 | Test MAE: 35.356510
  ZPVE (eV)       | Val MAE: 5.818143 | Test MAE: 5.628845
  U₀ (eV)         | Val MAE: 11513.451172 | Test MAE: 11167.009766
  U (eV)          | Val MAE: 11159.145508 | Test MAE: 10953.441406
  H (eV)          | Val MAE: 11825.337891 | Test MAE: 11572.991211
  G (eV)          | Val MAE: 10923.132812 | Test MAE: 10678.585938
  c_v             | Val MAE: 1.343932 | Test MAE: 1.304622
  U₀_atom         | Val MAE: 3.047005 | Test MAE: 2.952863
  U_atom          | Val MAE: 91.235138 | Test MAE: 89.480690
  H_atom          | Val MAE: 84.518341 | Test MAE: 81.678482
  G_atom          | Val MAE: 78.565903 | Test 

Train loss (MSE): 0.419109
  μ (D)           | Val MAE: 0.896204 | Test MAE: 0.910408
  α (Ang³)        | Val MAE: 0.507059 | Test MAE: 0.497382
  ε_HOMO (eV)     | Val MAE: 7.891164 | Test MAE: 7.856649
  ε_LUMO (eV)     | Val MAE: 13.676653 | Test MAE: 13.855630
  Δε (eV)         | Val MAE: 15.585810 | Test MAE: 15.494254
  ⟨R²⟩ (Ang²)     | Val MAE: 34.610268 | Test MAE: 34.101883
  ZPVE (eV)       | Val MAE: 5.230491 | Test MAE: 5.104712
  U₀ (eV)         | Val MAE: 11091.287109 | Test MAE: 10853.387695
  U (eV)          | Val MAE: 10822.551758 | Test MAE: 10633.606445
  H (eV)          | Val MAE: 11412.687500 | Test MAE: 11188.241211
  G (eV)          | Val MAE: 10348.962891 | Test MAE: 10169.496094
  c_v             | Val MAE: 1.376407 | Test MAE: 1.344350
  U₀_atom         | Val MAE: 3.159051 | Test MAE: 3.080045
  U_atom          | Val MAE: 85.839622 | Test MAE: 83.969414
  H_atom          | Val MAE: 84.847588 | Test MAE: 82.413353
  G_atom          | Val MAE: 77.142715 | Test 

Train loss (MSE): 0.386611
  μ (D)           | Val MAE: 0.914476 | Test MAE: 0.920354
  α (Ang³)        | Val MAE: 0.481086 | Test MAE: 0.472392
  ε_HOMO (eV)     | Val MAE: 7.327892 | Test MAE: 7.315070
  ε_LUMO (eV)     | Val MAE: 12.866349 | Test MAE: 13.024590
  Δε (eV)         | Val MAE: 14.702536 | Test MAE: 14.616063
  ⟨R²⟩ (Ang²)     | Val MAE: 37.071121 | Test MAE: 36.581039
  ZPVE (eV)       | Val MAE: 4.369573 | Test MAE: 4.238747
  U₀ (eV)         | Val MAE: 11141.192383 | Test MAE: 10949.930664
  U (eV)          | Val MAE: 10425.293945 | Test MAE: 10224.841797
  H (eV)          | Val MAE: 10456.375977 | Test MAE: 10189.246094
  G (eV)          | Val MAE: 10075.619141 | Test MAE: 9899.340820
  c_v             | Val MAE: 1.353522 | Test MAE: 1.323491
  U₀_atom         | Val MAE: 2.630508 | Test MAE: 2.565040
  U_atom          | Val MAE: 84.675064 | Test MAE: 82.949417
  H_atom          | Val MAE: 72.797829 | Test MAE: 70.509445
  G_atom          | Val MAE: 79.165092 | Test M

...

Train loss (MSE): 0.081242
  μ (D)           | Val MAE: 0.540589 | Test MAE: 0.539288
  α (Ang³)        | Val MAE: 0.096859 | Test MAE: 0.097084
  ε_HOMO (eV)     | Val MAE: 3.988915 | Test MAE: 3.985805
  ε_LUMO (eV)     | Val MAE: 6.092104 | Test MAE: 6.106084
  Δε (eV)         | Val MAE: 7.488369 | Test MAE: 7.426082
  ⟨R²⟩ (Ang²)     | Val MAE: 7.054037 | Test MAE: 6.857775
  ZPVE (eV)       | Val MAE: 1.178817 | Test MAE: 1.177666
  U₀ (eV)         | Val MAE: 2018.969849 | Test MAE: 1996.791382
  U (eV)          | Val MAE: 2086.650635 | Test MAE: 2002.924072
  H (eV)          | Val MAE: 2084.873291 | Test MAE: 2066.781982
  G (eV)          | Val MAE: 2080.583008 | Test MAE: 2045.284058
  c_v             | Val MAE: 0.277779 | Test MAE: 0.272462
  U₀_atom         | Val MAE: 0.608970 | Test MAE: 0.596114
  U_atom          | Val MAE: 16.850569 | Test MAE: 16.451897
  H_atom          | Val MAE: 16.663778 | Test MAE: 16.366413
  G_atom          | Val MAE: 15.386712 | Test MAE: 15.352844

Train loss (MSE): 0.081249
  μ (D)           | Val MAE: 0.540650 | Test MAE: 0.539329
  α (Ang³)        | Val MAE: 0.096885 | Test MAE: 0.097088
  ε_HOMO (eV)     | Val MAE: 3.988541 | Test MAE: 3.985502
  ε_LUMO (eV)     | Val MAE: 6.092196 | Test MAE: 6.106464
  Δε (eV)         | Val MAE: 7.488247 | Test MAE: 7.425721
  ⟨R²⟩ (Ang²)     | Val MAE: 7.055047 | Test MAE: 6.857208
  ZPVE (eV)       | Val MAE: 1.178820 | Test MAE: 1.177706
  U₀ (eV)         | Val MAE: 2019.165649 | Test MAE: 1996.473145
  U (eV)          | Val MAE: 2087.013428 | Test MAE: 2002.572998
  H (eV)          | Val MAE: 2085.774902 | Test MAE: 2066.317139
  G (eV)          | Val MAE: 2081.137451 | Test MAE: 2044.978516
  c_v             | Val MAE: 0.277802 | Test MAE: 0.272348
  U₀_atom         | Val MAE: 0.608926 | Test MAE: 0.595946
  U_atom          | Val MAE: 16.851927 | Test MAE: 16.450853
  H_atom          | Val MAE: 16.664028 | Test MAE: 16.364119
  G_atom          | Val MAE: 15.389734 | Test MAE: 15.352881

Train loss (MSE): 0.081246
  μ (D)           | Val MAE: 0.541122 | Test MAE: 0.539671
  α (Ang³)        | Val MAE: 0.096871 | Test MAE: 0.097113
  ε_HOMO (eV)     | Val MAE: 3.988801 | Test MAE: 3.985828
  ε_LUMO (eV)     | Val MAE: 6.092121 | Test MAE: 6.106173
  Δε (eV)         | Val MAE: 7.488630 | Test MAE: 7.425114
  ⟨R²⟩ (Ang²)     | Val MAE: 7.054037 | Test MAE: 6.861982
  ZPVE (eV)       | Val MAE: 1.179012 | Test MAE: 1.177672
  U₀ (eV)         | Val MAE: 2019.040405 | Test MAE: 1996.827026
  U (eV)          | Val MAE: 2086.773926 | Test MAE: 2003.898438
  H (eV)          | Val MAE: 2084.660645 | Test MAE: 2067.312988
  G (eV)          | Val MAE: 2080.740723 | Test MAE: 2046.000366
  c_v             | Val MAE: 0.277792 | Test MAE: 0.272428
  U₀_atom         | Val MAE: 0.609009 | Test MAE: 0.596188
  U_atom          | Val MAE: 16.854774 | Test MAE: 16.470715
  H_atom          | Val MAE: 16.664276 | Test MAE: 16.364374
  G_atom          | Val MAE: 15.389112 | Test MAE: 15.358128

Train loss (MSE): 0.081242
  μ (D)           | Val MAE: 0.541288 | Test MAE: 0.539810
  α (Ang³)        | Val MAE: 0.096864 | Test MAE: 0.097094
  ε_HOMO (eV)     | Val MAE: 3.988533 | Test MAE: 3.985506
  ε_LUMO (eV)     | Val MAE: 6.094440 | Test MAE: 6.109250
  Δε (eV)         | Val MAE: 7.491044 | Test MAE: 7.430899
  ⟨R²⟩ (Ang²)     | Val MAE: 7.054072 | Test MAE: 6.861084
  ZPVE (eV)       | Val MAE: 1.178893 | Test MAE: 1.177809
  U₀ (eV)         | Val MAE: 2019.101807 | Test MAE: 1996.466309
  U (eV)          | Val MAE: 2086.969482 | Test MAE: 2002.787598
  H (eV)          | Val MAE: 2085.191895 | Test MAE: 2066.754395
  G (eV)          | Val MAE: 2081.012695 | Test MAE: 2045.221313
  c_v             | Val MAE: 0.277803 | Test MAE: 0.272423
  U₀_atom         | Val MAE: 0.608955 | Test MAE: 0.595966
  U_atom          | Val MAE: 16.852915 | Test MAE: 16.463453
  H_atom          | Val MAE: 16.665228 | Test MAE: 16.369961
  G_atom          | Val MAE: 15.388520 | Test MAE: 15.356291

Train:  41%|████      | 354/860 [00:48<01:08,  7.44it/s]